In [ ]:
import pandas as pd
import json


def preprocess_semeval(file,index, is_test=False):
    with open(file, 'r') as f:
        data = json.load(f)

    processed_records = []
    for key, val in data.items():
        ending = val.get("ending") if val.get("ending") else ""

        input_text = (
            f"Definition: {val['judged_meaning']} [SEP] "
            f"Story: {val['precontext']} {val['sentence']} {ending}"
        )
        entry = dict({
            "id": int(key) + index,
            "text": input_text.strip(),
        })
        if not is_test:
            entry["label"] = float(val["average"])
            entry["stdev"] = val["stdev"]
        processed_records.append(entry)

    return processed_records

processed_records = preprocess_semeval('train.json', 0)
processed_records.extend(preprocess_semeval('dev.json', len(processed_records)))


df = pd.DataFrame(processed_records)

print(f"Shape: {df.shape}")
print(f"Example X: {df.iloc[0]['text']}")
print(f"Example Y: {df.iloc[0]['label']}")
df.head()

Shape: (2868, 4)
Example X: Definition: the difference in electrical charge between two points in a circuit expressed in volts [SEP] Story: The old machine hummed in the corner of the workshop. Clara examined its dusty dials with a furrowed brow. She wondered if it could be brought back to life. The potential couldn't be measured. She collected a battery reader and looked on earnestly, willing some life back into the old machine.
Example Y: 3.0


,id,text,label,stdev
0,0,Definition: the difference in electrical charg...,3.0,1.581139
1,1,Definition: the inherent capacity for coming i...,3.8,0.836660
2,2,Definition: the difference in electrical charg...,2.2,1.303840
3,3,Definition: the inherent capacity for coming i...,4.4,0.894427
4,4,Definition: the difference in electrical charg...,2.6,1.516575


In [ ]:

test_data = preprocess_semeval('test.json', 0, True)
df_test = pd.DataFrame(test_data)

print(f"Shape: {df_test.shape}")
print(f"Example X: {df_test.iloc[0]['text']}")
df_test.head()

Shape: (930, 2)
Example X: Definition: a structure consisting of a room or set of rooms at a single position along a vertical scale [SEP] Story: Maya had always been fascinated by the old mansion on the hill. She finally got a chance to explore it when the new owners offered guided tours. As she walked through the grand hallways, she marveled at how endless the building seemed. There were so many different levels to it. It was tiring walking up and down the stairs; the mansion spanned across five levels.


,id,text
0,0,Definition: a structure consisting of a room o...
1,1,Definition: an abstract place usually conceive...
2,2,Definition: a structure consisting of a room o...
3,3,Definition: an abstract place usually conceive...
4,4,Definition: a structure consisting of a room o...


In [ ]:
import torch
import numpy as np
from scipy.stats import spearmanr
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

model_name = "distilbert-base-uncased"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_func(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset_full = Dataset.from_pandas(df[["text", "label", "stdev"]])
train_val_split = train_dataset_full.train_test_split(test_size=0.15)

train_ds = train_val_split["train"].map(tokenize_func, batched=True)
val_ds = train_val_split["test"].map(tokenize_func, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.squeeze()

    spearman_rho, _ = spearmanr(predictions, labels)

    stdevs = np.array(val_ds["stdev"])
    diff = np.abs(predictions - labels)
    acc_within_sd = np.mean(diff <= stdevs)

    return {
        "spearman_rho": spearman_rho,
        "accuracy_within_sd": acc_within_sd
    }


model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1).to(device)

training_args = TrainingArguments(
    output_dir="./distilbert_semeval",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy_within_sd",
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

test_ds = Dataset.from_pandas(df_test[["text"]]).map(tokenize_func, batched=True)
preds = trainer.predict(test_ds)
df_test["prediction"] = preds.predictions.squeeze()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Epoch,Training Loss,Validation Loss,Spearman Rho,Accuracy Within Sd
1,1.394200,1.449410,0.155297,0.501160
2,1.350700,1.244681,0.393504,0.529002
3,1.150100,1.063513,0.529521,0.559165
4,0.847100,1.101763,0.529161,0.577726
5,0.642800,1.112598,0.543883,0.596288
6,0.622800,1.102630,0.562130,0.589327
7,0.380500,1.184952,0.551258,0.591647
8,0.346300,1.192889,0.564795,0.598608
9,0.346500,1.186672,0.567217,0.598608
10,0.234800,1.190369,0.563126,0.603248


KeyboardInterrupt: 

In [ ]:
df_test['final_prediction'] = df_test['prediction'].round().astype(int)

output_file = "predictions.jsonl"

with open(output_file, 'w') as f:
    for _, row in df_test.iterrows():
        entry = {
            "id": str(row['id']),
            "prediction": int(row['final_prediction'])
        }
        f.write(json.dumps(entry) + '\n')

print(f"File '{output_file}' has been created successfully.")


File 'predictions.jsonl' has been created successfully.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

def basic_tokenizer(text):
    return text.lower().replace("[sep]", "").split()

all_tokens = []
for text in df['text']:
    all_tokens.extend(basic_tokenizer(text))

word_counts = Counter(all_tokens)
vocab = {word: i+2 for i, (word, count) in enumerate(word_counts.items()) if count > 1}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

def encode_text(text, max_len=128):
    tokens = basic_tokenizer(text)
    encoded = [vocab.get(t, 1) for t in tokens[:max_len]]
    return encoded + [0] * (max_len - len(encoded))

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import requests
import zipfile
import os

def load_glove_manual(vocab, vector_dim=100):
    glove_path = f"glove.6B.{vector_dim}d.txt"
    zip_path = "glove.6B.zip"

    if not os.path.exists(glove_path):
        url = "https://nlp.stanford.edu/data/glove.6B.zip"
        response = requests.get(url, stream=True)
        with open(zip_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(".")
        print("Download and extraction complete.")

    embeddings_index = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs

    matrix_len = len(vocab)
    weights_matrix = np.zeros((matrix_len, vector_dim))
    words_found = 0

    for word, i in vocab.items():
        if word in embeddings_index:
            weights_matrix[i] = embeddings_index[word]
            words_found += 1
        else:
            weights_matrix[i] = np.random.normal(scale=0.6, size=(vector_dim,))

    print(f"Vocab size: {matrix_len} | GloVe matches: {words_found}")
    return torch.from_numpy(weights_matrix).float()

embedding_matrix = load_glove_manual(vocab, vector_dim=100)


Vocab size: 7440 | GloVe matches: 4769


In [ ]:
class PlausibilityLSTM(nn.Module):
    def __init__(self, embedding_matrix, embed_dim=100, hidden_dim=128, vocab_size=None):
        super(PlausibilityLSTM, self).__init__()
        # comments are to switch between glove and no glove and change the amount of layers
        # num_embeddings, embed_dim = embedding_matrix.shape

        # # Load the pre-trained GloVe weights
        # self.embedding = nn.Embedding.from_pretrained(
        #     embedding_matrix,
        #     freeze=False,
        #     padding_idx=0
        # )

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)

        self.fc_hidden = nn.Linear(hidden_dim * 2, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3) # Helps with generalization

        # self.fc_hidden2 = nn.Linear(64, 32)
        # self.relu2 = nn.ReLU()
        # self.dropout2 = nn.Dropout(0.1) # Helps with generalization

        # 4. Final Output Layer (Regression)
        self.fc_out = nn.Linear(64, 1)
        # self.fc_out = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (ht, _) = self.lstm(embedded)

        # Concatenate forward and backward final hidden states
        combined_h = torch.cat((ht[-2,:,:], ht[-1,:,:]), dim=1)

        # Pass through the new hidden layer
        x = self.fc_hidden(combined_h)
        x = self.relu(x)
        x = self.dropout(x)

        # x = self.fc_hidden2(x)
        # x = self.relu2(x)
        # x = self.dropout2(x)

        return self.fc_out(x).squeeze()

embedding_matrix=None


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import spearmanr
import numpy as np

def get_tensor_data(dataframe, vocab, is_test=False):
    X = torch.tensor([encode_text(t) for t in dataframe['text']])
    if is_test:
        return TensorDataset(X)
    y = torch.tensor(dataframe['label'].values).float()
    sd = torch.tensor(dataframe['stdev'].values).float()
    return TensorDataset(X, y, sd)

train_df = df.sample(frac=0.85, random_state=42)
val_df = df.drop(train_df.index)

train_loader = DataLoader(get_tensor_data(train_df, vocab), batch_size=16, shuffle=True)
val_loader = DataLoader(get_tensor_data(val_df, vocab), batch_size=16)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = PlausibilityLSTM(embedding_matrix, embed_dim=100, hidden_dim=128).to(device)
model = PlausibilityLSTM(embedding_matrix, embed_dim=100, hidden_dim=256, vocab_size=len(vocab)).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
# optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=0.90)

epochs = 50
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch_x, batch_y, _ in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    all_preds, all_labels, all_sds = [], [], []

    with torch.no_grad():
        for val_x, val_y, val_sd in val_loader:
            val_x = val_x.to(device)
            preds = model(val_x)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(val_y.numpy())
            all_sds.extend(val_sd.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_sds = np.array(all_sds)

    spearman, _ = spearmanr(all_preds, all_labels)
    acc_within_sd = np.mean(np.abs(all_preds - all_labels) <= all_sds)

    print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss/len(train_loader):.4f} | "
          f"Spearman: {spearman:.4f} | Acc_SD: {acc_within_sd:.4f}")

test_loader = DataLoader(get_tensor_data(df_test, vocab, is_test=True), batch_size=1)
model.eval()
test_preds = []
with torch.no_grad():
    for tx in test_loader:
        test_preds.append(model(tx[0].to(device)).item())

df_test['prediction'] = test_preds

Epoch 1/50 | Loss: 2.0636 | Spearman: 0.2090 | Acc_SD: 0.4860
Epoch 2/50 | Loss: 1.4413 | Spearman: 0.3202 | Acc_SD: 0.5047
Epoch 3/50 | Loss: 1.3462 | Spearman: 0.4051 | Acc_SD: 0.5233
Epoch 4/50 | Loss: 1.1631 | Spearman: 0.4465 | Acc_SD: 0.5512
Epoch 5/50 | Loss: 1.0897 | Spearman: 0.4782 | Acc_SD: 0.5488
Epoch 6/50 | Loss: 1.0563 | Spearman: 0.4912 | Acc_SD: 0.5651
Epoch 7/50 | Loss: 1.0478 | Spearman: 0.4974 | Acc_SD: 0.5605
Epoch 8/50 | Loss: 0.9278 | Spearman: 0.5287 | Acc_SD: 0.5628
Epoch 9/50 | Loss: 0.9306 | Spearman: 0.5049 | Acc_SD: 0.5628
Epoch 10/50 | Loss: 0.8730 | Spearman: 0.5269 | Acc_SD: 0.5674
Epoch 11/50 | Loss: 0.8700 | Spearman: 0.5254 | Acc_SD: 0.5953
Epoch 12/50 | Loss: 0.8672 | Spearman: 0.5216 | Acc_SD: 0.5930
Epoch 13/50 | Loss: 0.8126 | Spearman: 0.5255 | Acc_SD: 0.5884
Epoch 14/50 | Loss: 0.8199 | Spearman: 0.5136 | Acc_SD: 0.5907
Epoch 15/50 | Loss: 0.8123 | Spearman: 0.5215 | Acc_SD: 0.5884
Epoch 16/50 | Loss: 0.7889 | Spearman: 0.5274 | Acc_SD: 0.6023
E

No glove: <br>
Epoch 50/50 | Loss: 0.5488 | Spearman: 0.5385 | Acc_SD: 0.6163 <br>
No glove, 1 Dense relu layer with 64:<br>
Epoch 50/50 | Loss: 0.5599 | Spearman: 0.5373 | Acc_SD: 0.6070<br>
No glove, 2 Dense relu layers with 64, 32:<br>
Epoch 50/50 | Loss: 0.5595 | Spearman: 0.5400 | Acc_SD: 0.5930<br>
No Glove, 256 dim: <br>
Epoch 50/50 | Loss: 0.5114 | Spearman: 0.5421 | Acc_SD: 0.6000<br>
No Glove, 1 relu, 256 dim: <br>
Epoch 50/50 | Loss: 0.5726 | Spearman: 0.5601 | Acc_SD: 0.6116<br>
Glove:<br>
Epoch 50/50 | Loss: 0.1513 | Spearman: 0.4714 | Acc_SD: 0.6256<br>
Glove, 1 dense relu: <br>
Epoch 50/50 | Loss: 0.1654 | Spearman: 0.4722 | Acc_SD: 0.5837<br>
Glove, 1 dense relu, 64 dim: <br>
Epoch 50/50 | Loss: 0.2367 | Spearman: 0.5443 | Acc_SD: 0.6023<br>
Glove, 256 dim: <br>
Epoch 50/50 | Loss: 0.1411 | Spearman: 0.5124 | Acc_SD: 0.6140<br>
Glove, 1 relu, 256 dim: <br>
Epoch 50/50 | Loss: 0.2372 | Spearman: 0.4792 | Acc_SD: 0.6209<br>


In [ ]:
df_test['final_prediction'] = df_test['prediction'].round().astype(int)

output_file = "predictions.jsonl"

with open(output_file, 'w') as f:
    for _, row in df_test.iterrows():
        entry = {
            "id": str(row['id']),
            "prediction": int(row['final_prediction'])
        }
        f.write(json.dumps(entry) + '\n')

print(f"File '{output_file}' has been created successfully.")

File 'predictions.jsonl' has been created successfully.
